In [ ]:
"""
Simple & Effective Wheat Disease Detection with ConvNeXt

Simplified version keeping only the essentials:
✓ Multi-Scale Feature Fusion (main contribution)
✓ Clean, readable code (~400 lines vs 1700)
✓ Good performance with less complexity
✓ Easy to understand and modify
"""

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
from PIL import Image

# =============================================================================
# Configuration
# =============================================================================
DATASET_DIR = '../../dataset'
SAVE_DIR = '../../saved_models_and_data'
SPLIT_OUTPUT_DIR = '../../dataset_split'

IMAGE_SIZE = (320, 320)
BATCH_SIZE = 24
EPOCHS = 25
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 8

USE_MIXUP = True
MIXUP_ALPHA = 0.4

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(SPLIT_OUTPUT_DIR, exist_ok=True)

# =============================================================================
# Simple Focal Loss
# =============================================================================
class FocalLoss(nn.Module):
    """Focal Loss - focuses on hard examples"""
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

# =============================================================================
# MixUp Augmentation
# =============================================================================
def mixup_data(x, y, alpha=0.4):
    """MixUp - mixes two images together"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# =============================================================================
# Multi-Scale Fusion Module (Main Innovation)
# =============================================================================
class MultiScaleFusion(nn.Module):
    """
    Multi-scale feature fusion with 3 branches
    This is our main contribution - captures disease features at different scales
    """
    def __init__(self, channels):
        super().__init__()
        
        # Three branches: 3x3, 5x5, 7x7 convolutions
        self.branch1 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels//8),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels),
            nn.GELU()
        )
        
        self.branch2 = nn.Sequential(
            nn.Conv2d(channels, channels, 5, padding=2, groups=channels//8),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels),
            nn.GELU()
        )
        
        self.branch3 = nn.Sequential(
            nn.Conv2d(channels, channels, 7, padding=3, groups=channels//8),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels),
            nn.GELU()
        )
        
        # Fuse all branches
        self.fusion = nn.Sequential(
            nn.Conv2d(channels * 3, channels, 1),
            nn.BatchNorm2d(channels)
        )
    
    def forward(self, x):
        f1 = self.branch1(x)  # Fine details
        f2 = self.branch2(x)  # Medium patterns
        f3 = self.branch3(x)  # Large context
        
        # Concatenate and fuse
        concat = torch.cat([f1, f2, f3], dim=1)
        fused = self.fusion(concat)
        
        return fused

# =============================================================================
# Build Model
# =============================================================================
def build_model(num_classes):
    """Build ConvNeXt with Multi-Scale Fusion"""
    
    # Load pretrained ConvNeXt
    model = models.convnext_base(pretrained=True)
    in_features = model.classifier[2].in_features
    
    # Add our multi-scale fusion module
    model.fusion = MultiScaleFusion(in_features)
    
    # Simple but effective classifier
    model.classifier = nn.Sequential(
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.LayerNorm(in_features),
        nn.Linear(in_features, 512),
        nn.GELU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    
    # Custom forward pass
    def forward(x):
        x = model.features(x)     # ConvNeXt backbone
        x = model.fusion(x)       # Our multi-scale fusion
        x = model.classifier(x)   # Classification head
        return x
    
    model.forward = forward
    return model

# =============================================================================
# Dataset & DataLoader
# =============================================================================
class WheatDiseaseDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        self.samples = []
        for cls in self.classes:
            class_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(class_dir):
                continue
            for img_file in os.listdir(class_dir):
                if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    path = os.path.join(class_dir, img_file)
                    self.samples.append((path, self.class_to_idx[cls]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, target

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((int(IMAGE_SIZE[0] * 1.1), int(IMAGE_SIZE[1] * 1.1))),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def get_dataloaders():
    """Load or create data splits"""
    split_dirs = [os.path.join(SPLIT_OUTPUT_DIR, s) for s in ['train', 'val', 'test']]
    
    if all(os.path.isdir(d) for d in split_dirs):
        print("Loading existing data splits...")
        train_dataset = WheatDiseaseDataset(split_dirs[0], train_transform)
        val_dataset = WheatDiseaseDataset(split_dirs[1], test_transform)
        test_dataset = WheatDiseaseDataset(split_dirs[2], test_transform)
    else:
        print("Creating new data splits...")
        full_dataset = WheatDiseaseDataset(DATASET_DIR, train_transform)
        
        # Split data (70/15/15)
        indices = torch.randperm(len(full_dataset), generator=torch.Generator().manual_seed(42)).tolist()
        train_size = int(0.7 * len(full_dataset))
        val_size = int(0.15 * len(full_dataset))
        
        train_indices = indices[:train_size]
        val_indices = indices[train_size:train_size + val_size]
        test_indices = indices[train_size + val_size:]
        
        # Save splits to disk
        for split_name, split_indices in [('train', train_indices), ('val', val_indices), ('test', test_indices)]:
            for idx in split_indices:
                path, label = full_dataset.samples[idx]
                cls_name = full_dataset.classes[label]
                dest_dir = os.path.join(SPLIT_OUTPUT_DIR, split_name, cls_name)
                os.makedirs(dest_dir, exist_ok=True)
                shutil.copy(path, os.path.join(dest_dir, os.path.basename(path)))
        
        train_dataset = WheatDiseaseDataset(split_dirs[0], train_transform)
        val_dataset = WheatDiseaseDataset(split_dirs[1], test_transform)
        test_dataset = WheatDiseaseDataset(split_dirs[2], test_transform)
    
    # Weighted sampling for class balance
    targets = [s[1] for s in train_dataset.samples]
    class_counts = np.bincount(targets)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[t] for t in targets]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    
    # Create loaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    print(f"Data loaded: {len(train_dataset)} train, {len(val_dataset)} val, {len(test_dataset)} test")
    return train_loader, val_loader, test_loader, train_dataset.classes

# =============================================================================
# Training Function
# =============================================================================
def train_model(model, device, train_loader, val_loader, num_epochs=EPOCHS):
    """Simple but effective training loop"""
    
    criterion = FocalLoss(gamma=2.0)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    
    best_acc = 0.0
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    print("\nStarting training...")
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Apply MixUp augmentation
            if USE_MIXUP and np.random.rand() > 0.5:
                inputs, labels_a, labels_b, lam = mixup_data(inputs, labels, MIXUP_ALPHA)
                outputs = model(inputs)
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
        
        # Validation phase
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
        
        # Calculate metrics
        train_loss /= train_total
        train_acc = train_correct / train_total
        val_loss /= val_total
        val_acc = val_correct / val_total
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        scheduler.step()
        
        print(f"Epoch {epoch+1:2d}/{num_epochs} - "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")
        
        # Save best model
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, 'best_model_simple.pth'))
            print(f"  → New best! Val Acc: {val_acc:.4f}")
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    print(f"\nTraining complete! Best Val Acc: {best_acc:.4f}")
    return model, history

# =============================================================================
# Evaluation & Visualization
# =============================================================================
def evaluate_and_visualize(model, device, test_loader, class_names, history):
    """Evaluate model and create visualizations"""
    
    # Load best model
    model.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'best_model_simple.pth')))
    model.eval()
    
    # Evaluate on test set
    y_true, y_pred = [], []
    print("\nEvaluating on test set...")
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
    
    test_acc = np.mean(np.array(y_true) == np.array(y_pred))
    print(f"\n✓ Test Accuracy: {test_acc*100:.2f}%")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
    
    # Training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(history['train_loss'], label='Train', marker='o')
    ax1.plot(history['val_loss'], label='Val', marker='s')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot([a*100 for a in history['train_acc']], label='Train', marker='o')
    ax2.plot([a*100 for a in history['val_acc']], label='Val', marker='s')
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'training_curves_simple.png'), dpi=200)
    print("✓ Training curves saved")
    plt.show()
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, 
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted', fontsize=12, fontweight='bold')
    plt.ylabel('True', fontsize=12, fontweight='bold')
    plt.title(f'Confusion Matrix (Test Acc: {test_acc*100:.2f}%)', 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix_simple.png'), dpi=200)
    print("✓ Confusion matrix saved")
    plt.show()
    
    print(f"\n✓ All results saved to: {SAVE_DIR}")

# =============================================================================
# Main
# =============================================================================
if __name__ == '__main__':
    print("="*80)
    print("SIMPLE & EFFECTIVE WHEAT DISEASE DETECTION")
    print("="*80)
    
    # Setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}\n")
    
    # Load data
    train_loader, val_loader, test_loader, class_names = get_dataloaders()
    print(f"Classes: {class_names}\n")
    
    # Build model
    model = build_model(len(class_names)).to(device)
    num_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Model built with {num_params:.1f}M parameters\n")
    
    # Train
    model, history = train_model(model, device, train_loader, val_loader)
    
    # Evaluate
    evaluate_and_visualize(model, device, test_loader, class_names, history)
    
    print("\n" + "="*80)
    print("✓ COMPLETE!")
    print("="*80)



SIMPLE & EFFECTIVE WHEAT DISEASE DETECTION
Using device: cpu

Loading existing data splits...
Data loaded: 2621 train, 562 val, 562 test
Classes: ['aphid', 'army_worm', 'black_rust', 'brown_rust', 'common_rust', 'fusarium_head_blight', 'healthy', 'leaf_blight', 'powdery_mildew_leaf', 'spetoria', 'tan_spot', 'yellow_rust']



C:\Users\sayfs\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\sayfs\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ConvNeXt_Base_Weights.IMAGENET1K_V1`. You can also use `weights=ConvNeXt_Base_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model built with 95.1M parameters


Starting training...
